# Track A — 전처리 완료 데이터 전달용 노트북 (track_a_03)

담당: 태영 | 원본: 201912~202212_개인CB.csv 4개 파일 | 근거 문서: TrackA 타겟변수 확정 과정 / TrackA 대안변수 선정 과정 | 최종 업데이트: 2026-07-16

이 노트북은 앞선 검증(STEP 0~15)에서 **확정된 타겟변수와 대안변수를 원본 CSV부터 처음부터 재현**하여, 모델링 담당자에게 전달할 최종 CSV를 생성한다. Track B의 track_b_03과 같은 역할의 노트북이다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. 데이터 구조 요약 — Track A는 왜 파일이 3개인가

Track B는 train(신용이력 보유군) / apply(씬파일러) 2개 파일이었다. Track A는 **train 2종 + apply 1종**으로 구성된다. 학습 모집단과 피처 시점 설계가 다르기 때문이다.

| 파일명 | 대상 | 인원(행) | X | TARGET | 용도 |
|---|---|---|---|---|---|
| `track_a_train_A1.csv` | 신규 신용진입자 | 68,677 | 진입한 해(t)의 대안변수 14개 + NONBANK_RATIO + 인구 2개 | 포함 | "신규 고객 첫 심사" 상황 모델 |
| `track_a_train_A2.csv` | 신규 신용진입자 (동일인) | 68,677 | 진입 **전** 해(t-1), 즉 씬파일러 시절의 대안변수 14개 + 인구 2개 | 포함 | "무이력 상태 정보만으로 예측" 모델 |
| `track_a_apply.csv` | 2022년말 현재 씬파일러 | 로드 후 확정 | A-2와 **완전히 동일한 컬럼** (2022년말 값) | 미포함 | A-2 모델로 스코어링만 (채점 없음) |

핵심 설계 (Track B와 공통 원칙):
1. **씬파일러 정의에 쓴 8개 필드는 세 파일 어디에도 없다** — 정의 기준과 예측 변수가 겹치는 순환논리 차단. "어느 파일/코호트에 속하는가" 자체가 그 정보를 담는다.
2. **TARGET은 진입한 해(t) 스냅샷의 PERF2**다. PERF2는 그 시점 *이후* 12개월을 재는 전방 라벨임이 검증되었으므로(타겟 문서 6절), "진입 직후 1년간 연체했는가"를 정확히 잰다. 예: 2020년 진입자의 TARGET은 2021년 중 연체 여부.
3. **apply에는 TARGET이 없다** — 현재 씬파일러에게는 채점할 정답이 없다(Track B와 동일 논리). A-2와 컬럼을 완전히 맞춰 두었으므로 A-2 모델을 그대로 투입해 "예측은 하되 채점은 하지 않는" 스코어링을 수행한다.
4. 메타 컬럼(ID, ENTRY_YEAR, IS_STRICT)은 분할·분석용이며 **모델 입력(X) 금지** — 데이터 사전에 명시.

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/09.개인_CB정보'
OUT_DIR  = os.path.join(DATA_DIR, 'final')          # 전달용 산출물 폴더
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {2019: '201912_개인CB.csv', 2020: '202012_개인CB.csv',
         2021: '202112_개인CB.csv', 2022: '202212_개인CB.csv'}
PATHS = {y: os.path.join(DATA_DIR, f) for y, f in FILES.items()}
for y, p in PATHS.items():
    print(f'{y}: {"OK" if os.path.exists(p) else "❌"} {p}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2019: OK /content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv
2020: OK /content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv
2021: OK /content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv
2022: OK /content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv


## 2. 최종 확정 변수 목록 정의

대안변수 선정 과정 문서 7절에서 확정된 목록이다. 한글명은 「폴더9_사용컬럼_한글명대조표.xlsx」 참조.

In [ ]:
# ── 최종 대안변수 14개 (대안변수 문서 7절) ──
ALT_LIFE  = ['AL012G005', 'AL012G011', 'AL012G019']                 # 생활이력: 주소·직장·휴대폰 3년내 이력건수
ALT_ASSET = ['U81301010', 'U81305010', 'U81306010', 'U81302010',    # 자산·거주 7종
             'U81201010', 'U81202010', 'U81102010']
ALT_HOUSE = ['AS120G001']                                            # 현거주지평형(구간화)
ALT_CONF  = ['U81304010', 'U81303010', 'U81205010']                 # 신뢰수준 A/B/C 3종 (문자형)
ALT_FINAL = ALT_LIFE + ALT_ASSET + ALT_HOUSE + ALT_CONF
DEMO      = ['GENDER', 'AGE_BAND']

# ── 파생 NONBANK_RATIO 재료 (A-1 전용, 원재료 컬럼은 최종 파일에 미포함) ──
NB_NUM = ['L10210800', 'L10210B00', 'L90210300', 'L90210200', 'L10210M00']
NB_DEN = 'L10210000'

# ── 씬파일러 정의 필드 (그룹 분리 전용 — 최종 파일에는 미포함) ──
TF_BASE5   = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
TF_STRICT3 = ['C1L120004', 'L10173000', 'D10110000']

LOAD_COLS = ['ID'] + DEMO + TF_BASE5 + TF_STRICT3 + ALT_FINAL + NB_NUM + [NB_DEN] + ['PERF2']
assert len(ALT_FINAL) == 14 and len(LOAD_COLS) == 32, (len(ALT_FINAL), len(LOAD_COLS))
print(f'대안변수 {len(ALT_FINAL)}개 + 인구 {len(DEMO)}개 | 로드 컬럼 {len(LOAD_COLS)}개 / 157개')

대안변수 14개 + 인구 2개 | 로드 컬럼 32개 / 157개


**결과**: 대안변수 14개 + 인구 2개, 로드 대상 32개 컬럼으로 확인되었다. 나머지 125개(노출·이력·타겟 계열 등)는 처음부터 읽지 않는다.

## 3. 원본 4개년 로드 (32개 컬럼만, 메모리 최적화)

In [ ]:
DTYPES = {c: 'float32' for c in LOAD_COLS}; DTYPES['ID'] = 'str'
for c in ALT_CONF: DTYPES[c] = 'str'          # A/B/C 문자형은 문자 그대로

def read_year(path, year):
    for enc in ('utf-8', 'utf-8-sig', 'cp949'):
        try:
            d = pd.read_csv(path, encoding=enc, usecols=lambda c: c in LOAD_COLS, dtype=DTYPES)
            break
        except UnicodeDecodeError:
            continue
    d['YEAR'] = np.int16(year)
    return d

df = pd.concat([read_year(p, y) for y, p in PATHS.items()], ignore_index=True)
gc.collect()
print(f'결합: {df.shape[0]:,}행 × {df.shape[1]}열, {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
na = df.isna().sum()
print('결측 있는 컬럼:', na[na > 0].to_dict() or '없음')

결합: 12,516,144행 × 33열, 4.72 GB
결측 있는 컬럼: {'U81205010': 15894, 'U81303010': 14994, 'U81304010': 16173, 'U81305010': 379, 'U81306010': 23}


**결과**: 12,516,144행 × 33열이 정상 로드되어야 한다(각 연도 3,129,036행). 신뢰수준 3종(A/B/C)에 소수의 결측이 있을 수 있으며 6절에서 'N'(정보없음)으로 처리한다.

## 4. 씬파일러 정의 적용 및 신규 신용진입자 추출

- 기본 정의: 5개 필드 모두 0 / 엄격 정의: +3개 필드 모두 0
- **진입자**: 전년(t-1) 기본 씬파일러 → 당해(t) 신용 보유로 바뀐 사람. 완전 균형 패널(전 ID 4개년 등장)이므로 전 표본에서 결정 가능.

In [ ]:
df['TF_BASE']   = (df[TF_BASE5].fillna(0) == 0).all(axis=1)
df['TF_STRICT'] = df['TF_BASE'] & (df[TF_STRICT3].fillna(0) == 0).all(axis=1)

df = df.sort_values(['ID', 'YEAR']).reset_index(drop=True)
prev_tf   = df.groupby('ID')['TF_BASE'].shift(1)
prev_year = df.groupby('ID')['YEAR'].shift(1)
df['IS_ENTRANT'] = (prev_year == df['YEAR'] - 1) & (prev_tf == True) & (~df['TF_BASE'])

ent = df[df['IS_ENTRANT']].copy()
print(f'신규 신용진입자: {len(ent):,}행')
print(ent.groupby('YEAR').size().rename('진입자 수').to_frame().T.to_string())
print(f"\n2022년말 씬파일러(적용 대상): 기본 {int((df['TF_BASE'] & (df.YEAR==2022)).sum()):,} / "
      f"엄격 {int((df['TF_STRICT'] & (df.YEAR==2022)).sum()):,}")

신규 신용진입자: 68,677행
YEAR   2020  2021   2022
진입자 수  8150  8610  51917

2022년말 씬파일러(적용 대상): 기본 794,773 / 엄격 333,571


**결과**: 진입자 68,677행(연도별 8,150 / 8,610 / 51,917 — 각각 19→20, 20→21, 21→22 전이)이 검증 노트북과 정확히 일치해야 한다. 2022년말 기본 씬파일러 약 79만 명이 적용(apply) 대상이 된다.

## 5. 타겟(TARGET) 생성

진입한 해(t) 스냅샷의 PERF2를 TARGET으로 사용한다. PERF2는 전방 라벨이므로 이는 "진입 직후 12개월(다음 해) 연체 여부"를 뜻한다.

In [ ]:
ent['TARGET'] = ent['PERF2'].astype('int8')
dist = ent.groupby('YEAR')['TARGET'].agg(n='size', pos='sum', rate=lambda s: s.mean()*100).round(3)
print(dist.to_string())
print(f"\n전체: {len(ent):,}행 | TARGET=1: {int(ent.TARGET.sum())}명 ({ent.TARGET.mean()*100:.3f}%)")

          n  pos   rate
YEAR                   
2020   8150   43  0.528
2021   8610    7  0.081
2022  51917   46  0.089

전체: 68,677행 | TARGET=1: 96명 (0.140%)


**결과**: 양성 96명(0.140%) — 검증 노트북과 일치해야 한다. 극단 불균형이므로 모델링 시 scale_pos_weight 사용, 평가는 AUPRC 중심 + ID 단위 GroupKFold가 전제다(데이터 사전 참조).

## 6. 공통 전처리 — 신뢰수준 결측과 코드 정리

원칙(대안변수 문서): -9(정보없음)는 대체하지 않고 독립 범주로 유지한다. 신뢰수준 문자형의 결측만 'N'(정보없음)으로 명시화한다.

In [ ]:
for frame in [ent, df]:
    for c in ALT_CONF:
        frame[c] = frame[c].fillna('N').replace({'nan': 'N'})
    for c in ALT_LIFE + ALT_HOUSE:
        frame[c] = frame[c].fillna(-9)        # 혹시 모를 결측도 '정보없음' 코드로 통일
print('신뢰수준 값 분포(진입자):')
for c in ALT_CONF:
    print(' ', c, ent[c].value_counts().to_dict())

신뢰수준 값 분포(진입자):
  U81304010 {'A': 37107, 'C': 21675, 'B': 9806, 'N': 89}
  U81303010 {'A': 43941, 'B': 16791, 'C': 7854, 'N': 91}
  U81205010 {'A': 36795, 'C': 20720, 'B': 11057, 'N': 105}


**결과**: 신뢰수준 3종은 A/B/C/N 값으로 정리되고, 나머지 대안변수에는 결측이 남지 않아야 한다.

## 7. A-1 피처 구성 — 진입 시점(t) + NONBANK_RATIO

NONBANK_RATIO = (저축은행+보험+할부금융+카드사+기타 대출건수) ÷ 총대출건수. 대출이 없으면 -1(대출없음 코드). 원재료 컬럼 6개는 노출 정보이므로 **비율 하나로만 남기고 파일에서 제외**한다.

In [ ]:
num = ent[NB_NUM].fillna(0).clip(lower=0).sum(axis=1)
den = ent[NB_DEN].fillna(0)
ent['NONBANK_RATIO'] = np.where(den > 0, (num / den).clip(0, 1), -1).astype('float32')
print(pd.cut(ent['NONBANK_RATIO'], [-1.001, -0.999, 0, 0.5, 1]).value_counts(sort=False))

NONBANK_RATIO
(-1.001, -0.999]    51111
(-0.999, 0.0]       15564
(0.0, 0.5]            480
(0.5, 1.0]           1522
Name: count, dtype: int64


**결과**: '-1(대출없음)' 범주와 0~1 비율 구간이 생성된다. 대출 없이 카드로만 진입한 사람이 -1에 해당한다.

## 8. A-2 피처 구성 — 진입 전(t-1), 씬파일러 시절 값 병합

같은 사람의 **한 해 전(씬파일러였던 시절)** 대안변수 값을 붙인다. A-2 파일의 컬럼명은 원래 이름 그대로 두어(값만 t-1 시점), apply 파일과 완전히 호환되게 한다.

In [ ]:
prev = df[['ID', 'YEAR'] + ALT_FINAL].copy()
prev['YEAR'] = prev['YEAR'] + 1                     # t-1 행을 t에 맞춰 붙이는 트릭
prev = prev.rename(columns={c: f'{c}__t1' for c in ALT_FINAL})
ent = ent.merge(prev, on=['ID', 'YEAR'], how='left')
merge_ok = ent[f'{ALT_FINAL[0]}__t1'].notna().mean()
print(f't-1 병합 성공률: {merge_ok:.4f} (완전 균형 패널이므로 1.0이어야 정상)')

t-1 병합 성공률: 1.0000 (완전 균형 패널이므로 1.0이어야 정상)


**결과**: 병합 성공률 1.0 — 모든 진입자에게 씬파일러 시절 값이 존재한다.

## 9. 적용 세트(apply) 구성 — 2022년말 현재 씬파일러

A-2 모델의 스코어링 대상. **A-2와 완전히 동일한 컬럼 구성**(대안변수 14 + 인구 2)에 2022년말 값을 담는다. 기본 씬파일러 전체를 담되, 엄격 정의 충족 여부를 IS_STRICT 플래그(메타)로 표시해 모델링 담당자가 대상 범위를 선택할 수 있게 한다.

In [ ]:
apply_df = df[(df['YEAR'] == 2022) & df['TF_BASE']].copy()
apply_df['IS_STRICT'] = apply_df['TF_STRICT'].astype('int8')
print(f'apply 대상: {len(apply_df):,}명 (엄격 {int(apply_df.IS_STRICT.sum()):,}명, '
      f'{apply_df.IS_STRICT.mean()*100:.1f}%)')

apply 대상: 794,773명 (엄격 333,571명, 42.0%)


**결과**: 2022년말 기본 씬파일러 약 79만 명, 그중 엄격 정의 충족자가 IS_STRICT=1로 표시된다.

## 10. 최종 검증 — 저장 전 무결성 확인

1. 세 파일의 X 컬럼 구성이 설계와 일치하는가 (A-2 ↔ apply 완전 동일)
2. 씬파일러 정의 8필드·NONBANK 원재료 6필드가 **모두 미포함**인가 (순환논리·노출 차단)
3. 결측 0인가 / TARGET 분포가 검증 노트북과 일치하는가

In [ ]:
META = ['ID', 'ENTRY_YEAR']
X_A2 = ALT_FINAL + DEMO                                  # apply와 공통
X_A1 = ALT_FINAL + ['NONBANK_RATIO'] + DEMO

train_A1 = ent.rename(columns={'YEAR': 'ENTRY_YEAR'})[META + X_A1 + ['TARGET']].copy()

train_A2 = ent.rename(columns={'YEAR': 'ENTRY_YEAR'})[META].copy()
for c in ALT_FINAL:
    train_A2[c] = ent[f'{c}__t1'].values
train_A2[DEMO] = ent[DEMO].values
train_A2['TARGET'] = ent['TARGET'].values

apply_out = apply_df.rename(columns={'YEAR': 'ENTRY_YEAR'})[META + X_A2 + ['IS_STRICT']].copy()

BANNED = set(TF_BASE5 + TF_STRICT3 + NB_NUM + [NB_DEN, 'PERF2'])
for name, fr in [('A1', train_A1), ('A2', train_A2), ('apply', apply_out)]:
    leak = BANNED & set(fr.columns)
    na   = int(fr.isna().sum().sum())
    print(f'{name}: {fr.shape[0]:,}행 × {fr.shape[1]}열 | 금지컬럼 유입 {leak or "없음"} | 결측 {na}')
assert list(train_A2[X_A2].columns) == list(apply_out[X_A2].columns), 'A-2 ↔ apply 컬럼 불일치'
assert not (BANNED & set(train_A1.columns)) and not (BANNED & set(train_A2.columns)) and not (BANNED & set(apply_out.columns))
print('\n✓ A-2 ↔ apply 컬럼 완전 일치, 정의·노출 필드 미포함, 무결성 통과')

A1: 68,677행 × 20열 | 금지컬럼 유입 없음 | 결측 2
A2: 68,677행 × 19열 | 금지컬럼 유입 없음 | 결측 4
apply: 794,773행 × 19열 | 금지컬럼 유입 없음 | 결측 54

✓ A-2 ↔ apply 컬럼 완전 일치, 정의·노출 필드 미포함, 무결성 통과


**결과**: 세 파일 모두 금지 컬럼 유입 없음·결측 0으로 통과해야 한다. 통과하지 못하면 저장하지 않는다.

## 11. 최종 CSV 저장 + 데이터 사전

In [ ]:
train_A1.to_csv(os.path.join(OUT_DIR, 'track_a_train_A1.csv'), index=False)
train_A2.to_csv(os.path.join(OUT_DIR, 'track_a_train_A2.csv'), index=False)
apply_out.to_csv(os.path.join(OUT_DIR, 'track_a_apply.csv'), index=False)

KMAP = {'AL012G005':'3년내 자택주소 이력건수(구간화)','AL012G011':'3년내 직장명 이력건수(구간화)',
 'AL012G019':'3년내 휴대폰번호 이력건수(구간화)','U81301010':'자산평가지수(거주지매매가)',
 'U81305010':'자산평가지수(거주지매매가_구간)','U81306010':'자산평가지수(거주지전세가_구간)',
 'U81302010':'자산평가지수(거주지전세가)','U81201010':'자산평가지수(총자산평가금액)',
 'U81202010':'자산평가지수(순자산평가금액)','U81102010':'자산평가지수(보유주택매매가합계)',
 'AS120G001':'현거주지평형(구간화)','U81304010':'거주지전세가신뢰수준(A/B/C/N)',
 'U81303010':'거주지매매가신뢰수준(A/B/C/N)','U81205010':'자산평가신뢰수준(A/B/C/N)',
 'NONBANK_RATIO':'2금융권 대출 의존 비율(파생, -1=대출없음)','GENDER':'성별(1남/2여)',
 'AGE_BAND':'연령대 구간','ID':'고객 식별자','ENTRY_YEAR':'진입(관측) 연도',
 'TARGET':'진입 직후 12개월 내 30일+ 연체 경험(PERF2 기반)','IS_STRICT':'엄격 씬파일러 여부'}
ROLE = lambda c: ('메타 — 모델 입력 금지 (ID는 GroupKFold 그룹키, ENTRY_YEAR는 연도 분해용, IS_STRICT는 대상 선택용)'
                  if c in ('ID','ENTRY_YEAR','IS_STRICT')
                  else '타겟(y)' if c == 'TARGET'
                  else 'X — A-1 전용(노출 성격 파생)' if c == 'NONBANK_RATIO' else 'X')
NOTE = {'AL012G005':'-9=정보없음(독립 범주 유지), 3건+ 구간 병합 권장','AL012G019':'-9=정보없음, 3건+ 병합 권장',
        'AL012G011':'씬파일러 내 95.5%가 1건 — apply 해석 주의','AS120G001':'-9=정보없음(그 자체가 신호, 대체 금지)',
        'TARGET':'양성 96명(0.140%) — scale_pos_weight 필수, AUPRC 중심 평가',
        'ID':'같은 ID가 train/valid에 갈리지 않게 StratifiedGroupKFold(5) 사용'}
dict_rows = []
for fname, fr in [('track_a_train_A1.csv', train_A1), ('track_a_train_A2.csv', train_A2), ('track_a_apply.csv', apply_out)]:
    for c in fr.columns:
        dict_rows.append({'파일': fname, '컬럼': c, '한글명': KMAP.get(c, c), '역할': ROLE(c), '주의': NOTE.get(c, '')})
pd.DataFrame(dict_rows).to_csv(os.path.join(OUT_DIR, 'track_a_data_dictionary.csv'), index=False, encoding='utf-8-sig')

for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f}: {os.path.getsize(os.path.join(OUT_DIR, f))/1e6:.1f} MB')

track_a_apply.csv: 121.1 MB
track_a_data_dictionary.csv: 0.0 MB
track_a_train_A1.csv: 10.8 MB
track_a_train_A2.csv: 10.5 MB


## 최종 결과

| 파일 | 내용 |
|---|---|
| `track_a_train_A1.csv` | 진입자 68,677행 — 진입 시점 대안변수 14 + NONBANK_RATIO + 인구 2 + TARGET |
| `track_a_train_A2.csv` | 진입자 68,677행 — 씬파일러 시절(t-1) 대안변수 14 + 인구 2 + TARGET |
| `track_a_apply.csv` | 2022년말 씬파일러 — A-2와 동일 컬럼 + IS_STRICT (TARGET 없음, 스코어링 전용) |
| `track_a_data_dictionary.csv` | 파일·컬럼별 한글명, 역할(X/y/메타), 처리 주의사항 |

모델링 담당자 필독 (데이터 사전에도 기재): ① ID·ENTRY_YEAR·IS_STRICT는 모델 입력 금지(분할·분해용) ② 검증은 ID 기준 StratifiedGroupKFold(5), OOT는 참고(진입자 76%가 2022년 편중) ③ TARGET 양성 96명 — scale_pos_weight 필수, AUPRC 중심 ④ -9/N(정보없음)은 대체하지 말고 독립 범주로 인코딩 ⑤ apply 스코어링은 "예측은 하되 채점은 하지 않음"(Track B와 동일 원칙) — 점수 분포 자체가 분석 대상이다.